In [ ]:
# # STEP 1: Install required libraries
# !pip install -q transformers accelerate bitsandbytes

# # STEP 2: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# # STEP 3: Define model path
# model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# drive_cache_path = "/content/drive/MyDrive/tinyllama_cache"
# local_cache_path = "/content/tinyllama_cache"

# # STEP 4: Check if model already exists in Drive (persisted)
# import os
# if os.path.exists(drive_cache_path):
#     print("✔️ Found model in Google Drive. Copying to local cache...")
#     !cp -r "/content/drive/MyDrive/tinyllama_cache" "/content/"
# else:
#     print("❌ Model not found in Drive. Will download and then save to Drive.")

# # STEP 5: Load tokenizer and model (downloads if not in local cache)
# from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
# import torch

# tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=local_cache_path)

# model = AutoModelForCausalLM.from_pretrained(
#     model_id,
#     cache_dir=local_cache_path,
#     device_map="auto",
#     torch_dtype=torch.float16,
#     low_cpu_mem_usage=True
# )

# # STEP 6: Create pipeline
# llama_pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     # Removed device argument as accelerate handles it
# )

# # STEP 7: Save to Google Drive if not already saved
# if not os.path.exists(drive_cache_path):
#     print("✅ Saving model to Google Drive for future use...")
#     !cp -r "/content/tinyllama_cache" "/content/drive/MyDrive/"

# # STEP 8: Run a test prompt
# output = llama_pipe("What is Artificial Intelligence?", max_new_tokens=50)
# print(output[0]['generated_text'])


# output = pipeline(
#     "text-generation",
#     model = model,
#     tokenizer = tokenizer,
# )(
#     "What is tineLlama LLM",  # Pass the prompt as the first argument
#     max_new_tokens = 50
# )

# print(output)



# from datasets import Dataset
"""
import json

# ✅ Load the entire JSON array properly
with open("/content/drive/MyDrive/AIML/fineTune.json", "r") as f:
    data = json.load(f)

# ✅ Format into instruction-tuning prompt format
def format_alpaca(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n"
    if example.get("input"):  # handle optional input
        prompt += f"### Input:\n{example['input']}\n\n"
    prompt += f"### Response:\n{example.get('output', '')}"
    return {"text": prompt}

# ✅ Format all examples
formatted_data = [format_alpaca(sample) for sample in data]

# ✅ Convert to HuggingFace Dataset
dataset = Dataset.from_list(formatted_data)

# ✅ Preview first few samples
for i in range(min(5, len(dataset))):
    print(dataset[i]["text"])
    print("-" * 20)
"""

# hf_dataset = dataset
# hf_dataset.save_to_disk("/content/drive/MyDrive/AIML/hf_dataset")
"""

from datasets import load_from_disk

dataset_path = "/content/drive/MyDrive/AIML/hf_dataset"
hf_dataset = load_from_disk(dataset_path)
"""



"""
from transformers import AutoTokenizer

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",  # You can also try "longest"
        max_length=512
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = hf_dataset.map(tokenize, batched=True)
"""


"""
print(tokenized_dataset.column_names)
"""


"""
import pprint
pprint.pprint(tokenized_dataset[0])
"""


# tokenized_dataset = tokenized_dataset.remove_columns(["text"])
# print(tokenized_dataset[0])

# Save tokenized dataset to reuse later
"""
tokenized_dataset.save_to_disk("/content/drive/MyDrive/AIML/tokenized_dataset")
"""
!pip install -q transformers accelerate bitsandbytes evaluate rouge_score bert_score

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
❌ Model not found in Drive. Will download and then save to Drive.


Device set to use cuda:0


✅ Saving model to Google Drive for future use...
What is Artificial Intelligence?
Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that are typically performed by humans. It is a field of computer science that involves developing systems that can think, learn, and reason like humans do.


In [ ]:
from huggingface_hub import login
import os

# Load from Colab's secrets
hf_token = os.environ.get("HF_TOKEN")

# Log in using the token (recommended if you’ll push models or access private ones)
login(token=hf_token)


In [1]:
!pip install -q transformers accelerate bitsandbytes evaluate rouge_score bert_score gradio huggingface_hub

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.3 MB/s eta 0:00:00


In [2]:
# Step 1 ~ Mount your drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


| Field            | Meaning                                                                                                                            |
| ---------------- | ---------------------------------------------------------------------------------------------------------------------------------- |
| `input_ids`      | Tokenized version of the full prompt (instruction + optional input + response). These are integers representing vocabulary tokens. |
| `attention_mask` | Indicates which tokens are padding: `1` = real token, `0` = padding.                                                               |
| `labels`         | Target output for training. For basic fine-tuning, we set `labels = input_ids` so the model learns to predict the next token.      |


In [ ]:
# import json
# from datasets import Dataset

# # ✅ Load the entire JSON array properly
# with open("/content/drive/MyDrive/AIML/fineTune2.json", "r") as f:
#     data = json.load(f)

# # ✅ Format into instruction-tuning prompt format
# def format_alpaca(example):
#     prompt = f"### Instruction:\n{example['instruction']}\n\n"
#     if example.get("input"):  # handle optional input
#         prompt += f"### Input:\n{example['input']}\n\n"
#     prompt += f"### Response:\n{example.get('output', '')}"
#     return {"text": prompt}

# # ✅ Format all examples
# formatted_data = [format_alpaca(sample) for sample in data]

# # ✅ Convert to HuggingFace Dataset
# dataset = Dataset.from_list(formatted_data)

# # ✅ Preview first few samples
# for i in range(min(5, len(dataset))):
#     print(dataset[i]["text"])
#     print("-" * 20)

### Instruction:
What is Generative AI?

### Response:
Generative AI is a subset of artificial intelligence that creates new content, data, or outputs by learning patterns from existing datasets. Unlike discriminative models that classify or predict, generative models produce original text, images, audio, code, or other media. These systems use neural networks like GANs, VAEs, transformers, and diffusion models to understand data distributions and generate novel content that resembles the training data. Popular examples include ChatGPT for text, DALL-E for images, and GitHub Copilot for code generation.
--------------------
### Instruction:
How does Generative AI differ from traditional AI?

### Response:
Traditional AI focuses on analyzing, classifying, and making predictions from existing data using rule-based systems or discriminative models. Generative AI creates entirely new content by learning underlying data patterns and distributions. Traditional AI answers questions like 'Is t

In [ ]:
# from transformers import AutoTokenizer

# model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# tokenizer = AutoTokenizer.from_pretrained(model_id)

# def tokenize(example):
#     result = tokenizer(
#         example["text"],
#         truncation=True,
#         padding="max_length",  # You can also try "longest"
#         max_length=512
#     )
#     result["labels"] = result["input_ids"].copy()
#     # Remove the original 'text' column after tokenization
#     result.pop("text", None)
#     return result

# tokenized_dataset = dataset.map(tokenize, batched=True)
# tokenized_dataset = tokenized_dataset.remove_columns(["text"])

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

In [ ]:
# tokenized_dataset.save_to_disk("/content/drive/MyDrive/AIML/final_tokenized_dataset")

Saving the dataset (0/1 shards):   0%|          | 0/301 [00:00<?, ? examples/s]

In [3]:
from datasets import load_from_disk

# # ✅ Already tokenized, load it
tokenized_dataset = load_from_disk("/content/drive/MyDrive/AIML/final_tokenized_dataset")

from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import get_peft_model, LoraConfig
temp_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = temp_split["train"]
temp_eval = temp_split["test"].train_test_split(test_size=0.5, seed=42)
eval_dataset = temp_eval["train"]  # 10% for validation
test_dataset = temp_eval["test"]   # 10% for final testing

In [4]:
print(tokenized_dataset.column_names)
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}, Test: {len(test_dataset)}")

# Model setup with improved LoRA config
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(model_id)

['input_ids', 'attention_mask', 'labels']
Train: 240, Eval: 30, Test: 31


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
lora_config = LoraConfig(
    r=16,  # Reduced rank to prevent overfitting
    lora_alpha=32,  # Increased alpha for better learning
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # More modules
    lora_dropout=0.1,  # Increased dropout
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [ ]:
# !pip install evaluate rouge_score bert_score

In [5]:
import torch
import math
import evaluate
import numpy as np

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

def improved_compute_metrics(eval_pred):
    logits, labels = eval_pred

    # Handle numpy arrays properly
    if isinstance(logits, np.ndarray):
        logits = torch.from_numpy(logits)
    if isinstance(labels, np.ndarray):
        labels = torch.from_numpy(labels)

    # Move to CPU if needed
    logits = logits.cpu() if logits.is_cuda else logits
    labels = labels.cpu() if labels.is_cuda else labels

    # Calculate loss only on valid tokens
    shift_logits = logits[..., :-1, :].contiguous()
    shift_labels = labels[..., 1:].contiguous()

    # Flatten for loss calculation
    flat_logits = shift_logits.view(-1, shift_logits.size(-1))
    flat_labels = shift_labels.view(-1)

    # Create mask for valid tokens (not pad tokens)
    valid_mask = flat_labels != tokenizer.pad_token_id

    if valid_mask.sum() == 0:
        return {
            "eval_loss": float('inf'),
            "eval_perplexity": float('inf'),
            "eval_token_accuracy": 0.0,
            "eval_bleu": 0.0,
            "eval_rouge1": 0.0,
            "eval_rougeL": 0.0,
            "eval_bertscore_f1": 0.0
        }

    # Calculate loss
    loss = torch.nn.functional.cross_entropy(
        flat_logits[valid_mask],
        flat_labels[valid_mask],
        reduction='mean'
    )

    # Calculate perplexity
    perplexity = torch.exp(loss).item()

    # Generate predictions
    preds = shift_logits.argmax(dim=-1)

    # Token-level accuracy on valid tokens only
    correct = (preds.view(-1) == shift_labels.view(-1)) & valid_mask
    token_accuracy = correct.sum().item() / valid_mask.sum().item()

    # Decode for text-based metrics
    # Only decode non-empty sequences
    decoded_preds = []
    decoded_labels = []

    for i in range(preds.shape[0]):
        # Get prediction sequence
        pred_seq = preds[i]
        label_seq = shift_labels[i]

        # Find first pad token or use full sequence
        pred_mask = pred_seq != tokenizer.pad_token_id
        label_mask = label_seq != tokenizer.pad_token_id

        if pred_mask.sum() > 0:
            pred_text = tokenizer.decode(pred_seq[pred_mask], skip_special_tokens=True).strip()
        else:
            pred_text = ""

        if label_mask.sum() > 0:
            label_text = tokenizer.decode(label_seq[label_mask], skip_special_tokens=True).strip()
        else:
            label_text = ""

        if pred_text and label_text:  # Only include non-empty pairs
            decoded_preds.append(pred_text)
            decoded_labels.append(label_text)

    # Calculate text-based metrics only if we have valid sequences
    if decoded_preds and decoded_labels:
        try:
            # BLEU score
            bleu_result = bleu.compute(
                predictions=decoded_preds,
                references=[[label] for label in decoded_labels]
            )
            bleu_score = bleu_result["bleu"] if bleu_result else 0.0

            # ROUGE scores
            rouge_result = rouge.compute(
                predictions=decoded_preds,
                references=decoded_labels
            )
            rouge1_score = rouge_result["rouge1"] if rouge_result else 0.0
            rougeL_score = rouge_result["rougeL"] if rouge_result else 0.0

            # BERTScore for semantic similarity
            bert_result = bertscore.compute(
                predictions=decoded_preds,
                references=decoded_labels,
                model_type="distilbert-base-uncased"
            )
            bert_f1 = np.mean(bert_result["f1"]) if bert_result else 0.0

        except Exception as e:
            print(f"Error in text metrics calculation: {e}")
            bleu_score = 0.0
            rouge1_score = 0.0
            rougeL_score = 0.0
            bert_f1 = 0.0
    else:
        bleu_score = 0.0
        rouge1_score = 0.0
        rougeL_score = 0.0
        bert_f1 = 0.0

    return {
        "eval_loss": loss.item(),
        "eval_perplexity": perplexity,  # no Cap perplexity for display
        "eval_token_accuracy": token_accuracy,
        "eval_bleu": bleu_score,
        "eval_rouge1": rouge1_score,
        "eval_rougeL": rougeL_score,
        "eval_bertscore_f1": bert_f1
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/AIML/finallast_finetuned_model",
    per_device_train_batch_size=4,  # Increased batch size
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,  # Effective batch size = 8
    eval_strategy="steps",
    eval_steps=50,  # More frequent evaluation
    save_strategy="steps",
    save_steps=50,
    num_train_epochs=15,  # Reduced epochs to prevent overfitting
    learning_rate=1e-4,  # Slightly higher learning rate
    weight_decay=0.01,   # L2 regularization
    warmup_steps=500,    # Learning rate warmup
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",  # Use loss for early stopping
    greater_is_better=False,
    save_total_limit=3,  # Keep only best 3 checkpoints
    report_to="wandb",
    logging_dir="logs",
    fp16=True,  # Mixed precision training
    dataloader_drop_last=True,
    remove_unused_columns=False
)

# Custom Trainer with early stopping
class EarlyStoppingTrainer(Trainer):
    def __init__(self, *args, early_stopping_patience=5, early_stopping_threshold=0.001, **kwargs):
        super().__init__(*args, **kwargs)
        self.early_stopping_patience = early_stopping_patience
        self.early_stopping_threshold = early_stopping_threshold
        self.early_stopping_patience_counter = 0
        self.best_metric = None

    def evaluate(self, *args, **kwargs):
        metrics = super().evaluate(*args, **kwargs)

        # Early stopping logic
        current_metric = metrics.get("eval_loss")
        if current_metric is not None:
            if self.best_metric is None:
                self.best_metric = current_metric
            elif current_metric > self.best_metric - self.early_stopping_threshold:
                self.early_stopping_patience_counter += 1
                if self.early_stopping_patience_counter >= self.early_stopping_patience:
                    print(f"Early stopping at step {self.state.global_step}")
                    self.control.should_training_stop = True
            else:
                self.best_metric = current_metric
                self.early_stopping_patience_counter = 0

        return metrics

# Alternative: Use built-in early stopping callback
from transformers import EarlyStoppingCallback

# Option 1: Use custom trainer
trainer = EarlyStoppingTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=improved_compute_metrics,
    early_stopping_patience=5
)

# Option 2: Use built-in early stopping (comment out Option 1 and uncomment this)
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     tokenizer=tokenizer,
#     compute_metrics=improved_compute_metrics,
#     callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
# )

/tmp/ipython-input-4187159072.py:29: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `EarlyStoppingTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


In [ ]:
trainer.train()

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: riyagupta3825 (riyagupta3825-jecrc-foundation) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss,Perplexity,Token Accuracy,Bleu,Rouge1,Rougel,Bertscore F1
50,8.830900,7.887918,9.307999,0.531288,0.152804,0.487312,0.403969,0.839442
100,0.939600,0.902811,9.042598,0.530430,0.152083,0.487606,0.404671,0.839663
150,0.808500,0.775764,6.679093,0.582033,0.211833,0.534764,0.449274,0.854061
200,0.721800,0.734687,6.030936,0.594205,0.240047,0.543364,0.456221,0.860422
250,0.733000,0.717133,5.765240,0.599177,0.245838,0.540435,0.459947,0.862050
300,0.674700,0.704713,5.601713,0.603120,0.245732,0.545228,0.466419,0.863365
350,0.632400,0.702357,5.566379,0.606378,0.248677,0.550936,0.469902,0.864236
400,0.606900,0.714227,5.738250,0.601406,0.243811,0.543349,0.463689,0.863256
450,0.586100,0.712495,5.730610,0.601234,0.246033,0.544510,0.464043,0.864441


...

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

TrainOutput(global_step=450, training_loss=2.050665822558933, metrics={'train_runtime': 812.9909, 'train_samples_per_second': 4.428, 'train_steps_per_second': 0.554, 'total_flos': 1.14907076886528e+16, 'train_loss': 2.050665822558933, 'epoch': 15.0})

In [ ]:
print("Running final evaluation on test set...")
test_results = trainer.evaluate(eval_dataset=test_dataset)
print("Final Test Results:")
for k, v in test_results.items():
    print(f"{k}: {v:.4f}")

Running final evaluation on test set...


Final Test Results:
eval_loss: 0.7143
eval_perplexity: 5.8517
eval_token_accuracy: 0.6012
eval_bleu: 0.2398
eval_rouge1: 0.5498
eval_rougeL: 0.4593
eval_bertscore_f1: 0.8648
eval_runtime: 7.8726
eval_samples_per_second: 3.9380
eval_steps_per_second: 1.0160
epoch: 15.0000


In [ ]:
# model.save_pretrained("/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL")
# tokenizer.save_pretrained("/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL")

('/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL/tokenizer_config.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL/special_tokens_map.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL/chat_template.jinja',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL/tokenizer.model',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL/added_tokens.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL/tokenizer.json')

In [ ]:
# model.save_pretrained("/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(2)")
# tokenizer.save_pretrained("/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(2)")

('/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(2)/tokenizer_config.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(2)/special_tokens_map.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(2)/chat_template.jinja',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(2)/tokenizer.json')

In [ ]:
model.save_pretrained("/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)")
tokenizer.save_pretrained("/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)")

('/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)/tokenizer_config.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)/special_tokens_map.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)/chat_template.jinja',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)/tokenizer.model',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)/added_tokens.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)/tokenizer.json')

In [ ]:
# 1️⃣ Path to your last saved checkpoint
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
checkpoint_path = "/content/drive/MyDrive/AIML/finallast_finetuned_model/checkpoint-450"

# 2️⃣ Load tokenizer from checkpoint
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

# 3️⃣ Load base model from checkpoint
base_model = AutoModelForCausalLM.from_pretrained(checkpoint_path)

# # 4️⃣ Reapply LoRA config
# lora_config = LoraConfig(
#     r=16,  # Reduced rank to prevent overfitting
#     lora_alpha=32,  # Increased alpha for better learning
#     target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # More modules
#     lora_dropout=0.1,  # Increased dropout
#     bias="none",
#     task_type="CAUSAL_LM"
# )
# model = get_peft_model(base_model, lora_config)
# model.print_trainable_parameters()
model = PeftModel.from_pretrained(base_model, checkpoint_path, is_trainable=True)
model.train()
model.print_trainable_parameters()


/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [ ]:
# from transformers import TrainingArguments, Trainer
# training_args = TrainingArguments(
#     output_dir="/content/drive/MyDrive/AIML/finallast_finetuned_model_continued",
#     per_device_train_batch_size=1,  # Increased batch size
#     per_device_eval_batch_size=1,
#     gradient_accumulation_steps=2,  # Effective batch size = 8
#     eval_strategy="steps",
#     gradient_checkpointing=True,
#     eval_steps=100,  # More frequent evaluation
#     save_strategy="steps",
#     dataloader_pin_memory=False,
#     save_steps=100,
#     num_train_epochs=10,  # Reduced epochs to prevent overfitting
#     learning_rate=1e-4,  # Slightly higher learning rate
#     weight_decay=0.01,   # L2 regularization
#     warmup_steps=500,    # Learning rate warmup
#     logging_steps=10,
#     load_best_model_at_end=True,
#     metric_for_best_model="eval_loss",  # Use loss for early stopping
#     greater_is_better=False,
#     save_total_limit=3,  # Keep only best 3 checkpoints
#     report_to="wandb",
#     logging_dir="logs",
#     fp16=True,  # Mixed precision training
#     dataloader_drop_last=True,
#     remove_unused_columns=False,
#     resume_from_checkpoint=True  # <-- resume flag
# )

# # Custom Trainer with early stopping
# class EarlyStoppingTrainer(Trainer):
#     def __init__(self, *args, early_stopping_patience=5, early_stopping_threshold=0.001, **kwargs):
#         super().__init__(*args, **kwargs)
#         self.early_stopping_patience = early_stopping_patience
#         self.early_stopping_threshold = early_stopping_threshold
#         self.early_stopping_patience_counter = 0
#         self.best_metric = None

#     def evaluate(self, *args, **kwargs):
#         metrics = super().evaluate(*args, **kwargs)

#         # Early stopping logic
#         current_metric = metrics.get("eval_loss")
#         if current_metric is not None:
#             if self.best_metric is None:
#                 self.best_metric = current_metric
#             elif current_metric > self.best_metric - self.early_stopping_threshold:
#                 self.early_stopping_patience_counter += 1
#                 if self.early_stopping_patience_counter >= self.early_stopping_patience:
#                     print(f"Early stopping at step {self.state.global_step}")
#                     self.control.should_training_stop = True
#             else:
#                 self.best_metric = current_metric
#                 self.early_stopping_patience_counter = 0

#         return metrics

# # Alternative: Use built-in early stopping callback
# from transformers import EarlyStoppingCallback

# # Option 1: Use custom trainer
# trainer = EarlyStoppingTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=train_dataset,
#     eval_dataset=eval_dataset,
#     tokenizer=tokenizer,
#     compute_metrics=improved_compute_metrics,
#     early_stopping_patience=5
# )

# # Option 2: Use built-in early stopping (comment out Option 1 and uncomment this)
# # trainer = Trainer(
# #     model=model,
# #     args=training_args,
# #     train_dataset=train_dataset,
# #     eval_dataset=eval_dataset,
# #     tokenizer=tokenizer,
# #     compute_metrics=improved_compute_metrics,
# #     callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
# # )
# trainer.train()

/tmp/ipython-input-1271219489.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `EarlyStoppingTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: riyagupta3825 (riyagupta3825-jecrc-foundation) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Early stopping at step 600


TrainOutput(global_step=600, training_loss=0.5006311329205831, metrics={'train_runtime': 536.2252, 'train_samples_per_second': 4.476, 'train_steps_per_second': 2.238, 'total_flos': 3830235896217600.0, 'train_loss': 0.5006311329205831, 'epoch': 5.0})

In [ ]:
# print("Running final evaluation on test set...")
# test_results = trainer.evaluate(eval_dataset=test_dataset)
# print("Final Test Results:")
# for k, v in test_results.items():
#     print(f"{k}: {v:.4f}")

Running final evaluation on test set...


Early stopping at step 600
Final Test Results:
eval_loss: 0.7343
eval_perplexity: 6.3587
eval_token_accuracy: 0.5922
eval_bleu: 0.2333
eval_rouge1: 0.5368
eval_rougeL: 0.4555
eval_bertscore_f1: 0.8620
eval_runtime: 10.1184
eval_samples_per_second: 3.0640
eval_steps_per_second: 3.0640
epoch: 5.0000


In [ ]:
model.save_pretrained("/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST_re)")
tokenizer.save_pretrained("/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST_re)")

('/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST_re)/tokenizer_config.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST_re)/special_tokens_map.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST_re)/chat_template.jinja',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST_re)/tokenizer.model',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST_re)/added_tokens.json',
 '/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST_re)/tokenizer.json')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# 1. Load base model and tokenizer
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_model_path = "/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)"

tokenizer = AutoTokenizer.from_pretrained(adapter_model_path)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype=torch.float16, device_map="auto")

# 2. Load LoRA fine-tuned weights (PEFT model)
model = PeftModel.from_pretrained(base_model, adapter_model_path)
model.eval()  # Set model to evaluation mode

# 3. Define a user query for testing
user_query = "### Instruction: What are the main types of Generative AI models???\n### Response:"

# 4. Tokenize the input prompt
inputs = tokenizer(user_query, return_tensors="pt").to("cuda")

# 5. Generate prediction
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=True,
        top_k=50,
        top_p=0.95,
        temperature=0.6,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id
    )

# 6. Decode and print output
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Model Response:\n")
print(response)

Model Response:

### Instruction: What are the main types of Generative AI models???
### Response: Various generative models exist, including generative adversarial networks (GANs), generative model-based systems (e.g., GPT-3), and transformer-based architectures like Transformers for Language Modeling (T5). Some use unsupervised learning techniques to learn from large datasets and create new concepts, while others focus on improving existing knowledge bases. Challenges include language translation quality, cultural appropriateness, and accessibility. Recent advances include hierarchical models that combine multiple generators to improve diversity and creativity. Promising directions include meta-learning for generative model training and self-critique systems that help agents improve their own models.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import os
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
drive_cache_path = "/content/drive/MyDrive/tinyllama_cache"
local_cache_path = "/content/tinyllama_cache"

tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=local_cache_path)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    cache_dir=local_cache_path,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

# STEP 6: Create pipeline
llama_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    # Removed device argument as accelerate handles it
)

# STEP 7: Save to Google Drive if not already saved
if not os.path.exists(drive_cache_path):
    print("✅ Saving model to Google Drive for future use...")
    !cp -r "/content/tinyllama_cache" "/content/drive/MyDrive/"

# STEP 8: Run a test prompt
output = llama_pipe(
    "What are the main types of Generative AI models???",
    max_new_tokens=100,
    temperature=0.7,    # Adds creativity
    top_p=0.9,          # Nucleus sampling
    do_sample=True,     # Enables sampling instead of greedy decoding
    repetition_penalty=1.1
)
print(output[0]["generated_text"])


Device set to use cuda:0


What are the main types of Generative AI models???


In [ ]:
# STEP 1: Install dependencies
!pip install gradio peft -q

In [6]:
# STEP 1: Imports
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import gradio as gr

# STEP 2: Load Model & Tokenizer
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_model_path = "/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)"

tokenizer = AutoTokenizer.from_pretrained(adapter_model_path)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_model_path)
model.eval()

# STEP 3: Chat Function
def chat_with_model(user_query, max_new_tokens=250, temperature=0.6, top_p=0.95):
    # Format input using chat template
    messages = [{"role": "user", "content": user_query}]
    user_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    # Tokenize and move to GPU/CPU
    inputs = tokenizer(user_prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=top_p,
            temperature=temperature,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id
        )

    # Decode tokens to text
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # 🚨 Remove special role tokens
    for token in ["<|assistant|>", "<|user|>", "<|system|>"]:
        response = response.replace(token, "")

    # 🚨 Remove the original user query if it appears in the output
    response = response.replace(user_query, "").strip()

    # 🚨 Remove the full prompt (to be extra safe)
    response = response.replace(user_prompt, "").strip()

    return response

# STEP 4: Build Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## Fine-tuned TinyLLaMA Chatbot (LoRA Adapter)")

    with gr.Row():
        # Left panel: Sliders stacked vertically
        with gr.Column(scale=1):
            max_new_tokens = gr.Slider(50, 500, value=250, step=10, label="Max New Tokens")
            temperature = gr.Slider(0.1, 1.5, value=0.6, step=0.1, label="Temperature")
            top_p = gr.Slider(0.1, 1.0, value=0.95, step=0.05, label="Top-p (Nucleus Sampling)")

        # Right panel: Query + Output
        with gr.Column(scale=3):
            user_query = gr.Textbox(
                lines=1,
                label="Enter your question",
                placeholder="Type your question and press Enter to generate response..."
            )
            output = gr.Textbox(lines=10, label="Model Response")
            submit_btn = gr.Button("Generate Response")

    # This will trigger when Enter is pressed in the textbox
    user_query.submit(
        fn=chat_with_model,
        inputs=[user_query, max_new_tokens, temperature, top_p],
        outputs=output,
        # Show the button click animation when Enter is pressed
        api_name="generate"
    )

    # Button action
    submit_btn.click(
        fn=chat_with_model,
        inputs=[user_query, max_new_tokens, temperature, top_p],
        outputs=output,
        api_name="generate_button"
    )

# STEP 5: Launch Gradio App
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7d643c261e577229e1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
from huggingface_hub import notebook_login

# Login (you’ll get a token from https://huggingface.co/settings/tokens)
notebook_login()

In [10]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

adapter_model_path = "/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)"
repo_name = "Riya012/TinyLlama_AI_ML_Chatbot"  # <- choose this

# Push adapter
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.push_to_hub(repo_name)

model = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
peft_model = PeftModel.from_pretrained(model, adapter_model_path)
peft_model.push_to_hub(repo_name)

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  /tmp/tmplf_mcgui/tokenizer.model      : 100%|##########|  500kB /  500kB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...p2axm8pvn/adapter_model.safetensors: 100%|##########| 18.0MB / 18.0MB            

CommitInfo(commit_url='https://huggingface.co/Riya012/TinyLlama_AI_ML_Chatbot/commit/1d3772e799c7e18fadc69e9d88c66114d4763ccb', commit_message='Upload model', commit_description='', oid='1d3772e799c7e18fadc69e9d88c66114d4763ccb', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Riya012/TinyLlama_AI_ML_Chatbot', endpoint='https://huggingface.co', repo_type='model', repo_id='Riya012/TinyLlama_AI_ML_Chatbot'), pr_revision=None, pr_num=None)

In [9]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel

# base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# adapter_model_path = "/content/drive/MyDrive/AIML/FINAL_FINETUNE_MODEL(LAST)"

# # Load
# base_model = AutoModelForCausalLM.from_pretrained(base_model_id, torch_dtype="float16")
# model = PeftModel.from_pretrained(base_model, adapter_model_path)

# # Merge LoRA
# merged_model = model.merge_and_unload()

# # Save & push merged model
# save_path = "./merged_tinyllama"
# merged_model.save_pretrained(save_path)
# tokenizer = AutoTokenizer.from_pretrained(base_model_id)
# tokenizer.save_pretrained(save_path)

# # Push to HF Hub
# from huggingface_hub import HfApi, notebook_login
# notebook_login()
# from huggingface_hub import upload_folder
# api = HfApi()
# api.create_repo(
#     name="tinyllama-finetuned-merged",
#     token=True,          # uses your logged-in token
#     repo_type="model",
#     private=False        # set True if you want it private
# )

# upload_folder(
#     repo_id="Riya012/tinyllama-finetuned-merged",
#     folder_path=save_path,
#     repo_type="model"
# )